[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-4_%EC%A2%85%ED%95%A9%EC%8B%A4%EC%8A%B5_%EC%A7%84%EB%8B%A8%EC%8B%9C%EC%8A%A4%ED%85%9C_%EC%9A%B4%EC%98%81%EC%A0%90%EA%B2%80.ipynb)

# 2-4 · 종합 실습 — 진단 모델을 현장에 넣기 전에 점검할 것

**⏱ 90분** · 모듈 2 마무리: 이상탐지 → 고장진단 → **운영**

## 🎯 이번 시간의 질문

> 정확도 97%짜리 진단 모델을 만들었습니다. **이제 현장에 바로 넣어도 될까요?**

실험실 정확도가 높아도 현장에서는 다른 문제가 생깁니다. 이번 시간에는 진단 시스템을 운영한다고 생각하고 세 가지를 점검합니다.

1. **네 부품을 한꺼번에** 진단하면 전체적으로 얼마나 맞히나?
2. 모델이 **확신하지 못할 때**는 어떻게 할까? (자동 판정 vs 사람 확인)
3. **센서가 조금 틀어지면**(영점 오차) 진단은 어떻게 될까?

이번 시간은 **과제 비중이 큽니다.** 앞 시간에 입력한 패턴을 스스로 조합해 보세요.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
raw = np.load('data/hydraulic_raw_4sensors.npz')
data = labels.merge(features, on='cycle_id')
data['PS1_peak9'] = raw['PS1'][:, 900:1000].mean(axis=1)      # 2-1에서 만든 밸브 특징

cols = ['PS1_mean', 'PS2_mean', 'PS3_mean', 'PS4_mean', 'PS5_mean', 'PS6_mean', 'EPS1_mean',
        'FS1_mean', 'FS2_mean', 'TS1_mean', 'TS2_mean', 'TS3_mean', 'TS4_mean', 'VS1_mean', 'PS1_peak9']
train = data[data['split'] == 'train']
test = data[data['split'] == 'test']

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
print('준비 완료 · 입력 특징', len(cols), '개')

## 1. 네 부품 진단 모델을 한 번에 만들기

2-3에서 부품마다 따로 쓰던 **P5 세 줄**을 `for` 반복문 안에 넣으면 네 부품의 모델을 한 번에 만들 수 있습니다.
만든 모델은 딕셔너리 `models`에, 예측 결과는 표 `diag`에 모읍니다.

- `models[부품] = 모델` : 부품 이름으로 모델을 꺼내 쓸 수 있게 저장
- `diag[부품] = 예측` : 부품마다 예측 결과를 열로 추가

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
parts = ['cooler', 'valve', 'pump_leakage', 'accumulator']
models = {}
diag = pd.DataFrame(index=test.index)

for part in parts:
    model = RandomForestClassifier(random_state=0)
    model.fit(train[cols], train[___])
    models[part] = model
    diag[part] = model.predict(___)
    print(part, '정확도:', round(accuracy_score(test[part], diag[part]) * 100, 1), '%')

✅ **체크포인트** — 냉각기 100.0%, 밸브 89.3%, 펌프 96.7%, 축압기 86.0%.

💬 **결과 해설** — 부품마다 난이도가 다릅니다. 냉각기는 완벽하지만 축압기는 7번 중 1번꼴로 틀립니다.
그렇다면 **한 사이클의 네 부품을 모두 맞힌** 비율은 얼마일까요? 각 부품 예측이 정답과 같은지를 표로 비교하고, 행마다 모두 참인지(`all(axis=1)`) 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
correct = diag == test[parts]
all_correct = correct.all(axis=1)
print('네 부품 모두 맞힌 사이클:', round(all_correct.mean() * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 부품별로는 86~100%인데, **네 개를 모두 맞힌 사이클은 72.9%**뿐입니다. 부품이 여러 개면 어딘가 하나는 틀릴 확률이 커집니다.
현장 보고서에 "정확도 97%"라고만 쓰면 오해를 부릅니다. **무엇에 대한 정확도인지** 밝혀야 합니다.

실제 진단 보고서처럼 몇 사이클의 예측과 정답을 나란히 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
report = diag.copy()
report.columns = ['냉각기(예측)', '밸브(예측)', '펌프누설(예측)', '축압기(예측)']
for part, name in zip(parts, ['냉각기(정답)', '밸브(정답)', '펌프누설(정답)', '축압기(정답)']):
    report[name] = test[part]
report.insert(0, 'cycle_id', test['cycle_id'])
report.sample(6, random_state=3)

## 2. 모델이 확신하지 못할 때 — 확률과 확신도

랜덤 포레스트는 `predict_proba()`로 **상태별 확률**(나무 100그루 중 그 상태에 투표한 비율)을 알려 줍니다.
가장 큰 확률을 **확신도**라고 합시다. 나무들의 의견이 한쪽으로 모이면 확신도가 1에 가깝고, 갈리면 낮아집니다.

가장 어려웠던 **축압기**로 확인합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
proba = models['accumulator'].predict_proba(test[cols])
confidence = proba.max(axis=1)
right = correct['accumulator']

plt.hist(confidence[right], bins=20, alpha=0.6, label='맞힌 사이클')
plt.hist(confidence[~right], bins=20, alpha=0.6, label='틀린 사이클')
plt.xlabel('확신도 (가장 큰 확률)')
plt.ylabel('사이클 수')
plt.legend()
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 맞힌 사이클은 대부분 확신도가 0.8~1.0이고, **틀린 사이클은 확신도가 낮은 쪽**에 몰려 있습니다.
그렇다면 운영 규칙을 이렇게 만들 수 있습니다: **"확신도 0.7 이상이면 자동 판정, 미만이면 정비원이 직접 확인"**.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
auto = confidence >= 0.7
print('자동 판정 비율:', round(auto.mean() * 100, 1), '%')
print('자동 판정한 사이클의 정확도:', round(right[auto].mean() * 100, 1), '%')
print('사람 확인으로 넘긴 사이클의 정확도:', round(right[~auto].mean() * 100, 1), '%')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 자동 판정 75.3%, 그 정확도 95.7%. 사람 확인으로 넘긴 쪽의 정확도는 56.6%.

💬 **결과 해설** — 전체 정확도는 86%지만, **확신 있는 75%만 자동 판정하면 정확도가 95.7%**로 올라갑니다. 나머지 25%는 모델도 헷갈리는 사이클이므로 사람이 확인하는 것이 맞습니다.
AI 진단을 "전부 자동" 또는 "쓸모없음"으로 보지 않고, **모델이 잘하는 부분만 맡기는 설계**가 현장 도입의 현실적인 방법입니다.

---
## 3. 실습 과제 (45분)

### 📝 과제 1 · 확신도 기준 정하기
축압기 진단에서 확신도 기준을 **0.6, 0.7, 0.8**로 바꿔 아래 표를 채우세요. 정비원이 하루에 확인할 수 있는 양이 한정되어 있다면 어떤 기준을 고르겠습니까?

| 기준 | 자동 판정 비율(%) | 자동 판정 정확도(%) | 사람 확인 비율(%) | 틀린 47건 중 사람에게 넘어간 수 |
|---|---|---|---|---|
| 0.6 | | | | |
| 0.7 | 75.3 | 95.7 | 24.7 | |
| 0.8 | | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for th in [0.6, 0.7, 0.8]: auto = confidence >= th  → 비율은 auto.mean(), 틀린 것 중 넘어간 수는 (~right & ~auto).sum()


✏️ **나의 답:**

### 📝 과제 2 · 다른 부품에도 적용
같은 분석을 **밸브**와 **펌프 누설**에 적용하세요(기준 0.7). 틀린 사이클이 확신도 낮은 쪽에 몰리는 현상이 똑같이 나타나나요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 2절 코드에서 'accumulator'를 'valve' 또는 'pump_leakage'로 바꿉니다. for part in ['valve', 'pump_leakage']: 로 반복해도 됩니다.


### 📝 과제 3 · 센서 영점이 조금 틀어진다면?
현장 센서는 시간이 지나면 **영점이 조금씩 어긋납니다**(드리프트). 유량계 `FS1`이 실제보다 **0.05, 0.1, 0.2, 0.3 L/min** 크게 읽힌다고 가정하고, 펌프 누설 진단 정확도가 어떻게 변하는지 그래프로 그리세요.

⚠️ 원래 `test`를 바꾸지 않도록 `.copy()`로 복사본을 만든 뒤 값을 더합니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: for b in [0, 0.05, 0.1, 0.2, 0.3]:
# X = test[cols].copy();  X['FS1_mean'] = X['FS1_mean'] + b
# 정확도 = accuracy_score(test['pump_leakage'], models['pump_leakage'].predict(X))


✏️ **나의 답:** 0.1 L/min(유량의 약 1.5%) 오차에서 정확도는? 왜 이렇게 민감할까요? (2-3의 결정나무 규칙을 떠올려 보세요) 현장에서는 어떤 대책이 필요할까요?

### 📝 과제 4 · 진단 시스템 운영 제안서 (5문장)
지금까지의 결과(2-2 ~ 2-4)를 근거로, 이 유압 장치에 **어떤 진단 시스템을 어떻게 운영할지** 5문장 이내로 제안하세요. 아래 항목을 포함합니다.
① 이상탐지와 진단을 각각 어디에 쓸지 ② 자동 판정과 사람 확인의 기준 ③ 센서 관리 ④ 모델을 다시 학습해야 하는 상황

✏️ **나의 답:**

---
## ✅ 모듈 2 정리

| 단계 | 한 일 | 핵심 |
|---|---|---|
| 2-1 특징 | 6,000개 값 → 평균·표준편차 등 | 좋은 특징은 상태에 따라 뚜렷이 갈린다. 특정 구간에 숨은 특징도 있다 |
| 2-2 이상탐지 | 정상만으로 z-점수·Isolation Forest | 고장 데이터 없이도 경보 가능. 미세 고장은 놓친다 |
| 2-3 진단 | 정답으로 결정나무·랜덤 포레스트 학습 | 무엇이 고장인지 판정. **좋은 특징 > 복잡한 모델** |
| 2-4 운영 | 전체 정확도·확신도·센서 오차 점검 | 애매하면 사람에게. 센서가 틀어지면 모델도 틀린다 |

지금까지는 "**지금** 어떤 상태인가"를 판단했습니다. 정비 계획에는 "**앞으로 얼마나** 버틸까"도 필요합니다.
→ 모듈 3에서는 필터가 막혀 가는 과정을 보고 **잔여수명(RUL)**을 예측합니다.